<!-- <style> * { font-size: 100%; } </style> -->
<small> 

# rsyncd
- https://linux.die.net/man/5/rsyncd.conf
- https://superuser.com/questions/243656/how-to-configure-and-use-rsyncd


</small>

In [ ]:
## Setup Working Directory & Files

# Working Dir
WORKING_DIRECTORY="/mnt/user/docker/rsyncd-server"
mkdir -p ${WORKING_DIRECTORY}
cd ${WORKING_DIRECTORY}

# Make Folders & Files
# mkdir -p ./{config,repositories}
mkdir -p ./{config,secrets}
ls -alt ${WORKING_DIRECTORY}

# Create Needed Files
touch ${WORKING_DIRECTORY}/secrets/rsyncd.secrets

# Docker

## Docker Context

In [ ]:
# Setup Docker on Remote Host via Context
# docker context create lxc-docker-01 --docker "host=ssh://lxc-docker-01"

set -euo pipefail

# SSH Alias Name (/config/.ssh/config)
CONTEXTS=(
  "nas-00"
  "nas-01"
  "nas-03"
  "lxc-docker-01"
  "lxc-docker-02"
  "lxc-docker-03"
  "lxc-docker-04"
  "lxc-docker-05"
  "lxc-docker-06"
  "lxc-docker-07"
  "lxc-docker-08"
  "lxc-docker-09"
)

for CONTEXT in "${CONTEXTS[@]}"; do
  if docker context inspect "${CONTEXT}" >/dev/null 2>&1; then
    printf 'Context %s : already exists\n' "${CONTEXT}"
  else
    docker context create "${CONTEXT}" \
      --docker "host=ssh://${CONTEXT}" \
      >/dev/null 2>&1
    printf 'Context %s : created successfully\n' "${CONTEXT}"
  fi
done

In [ ]:
# Remove Context
# docker context remove lxc-docker-06
# docker context remove lxc-docker-07

In [ ]:
# Use Context
# docker context use default
# docker context use lxc-docker-01
docker context use nas-03

## Build and Deploy Container

In [ ]:
# Build
docker compose --file docker-compose.yaml --env-file docker-compose.example.env build

In [ ]:
# Setup
WORKING_DIRECTORY="/mnt/user/docker/rsyncd-server"
mkdir -p ${WORKING_DIRECTORY}
cd ${WORKING_DIRECTORY}

# mkdir -p ./{secrets}
mkdir -p ./secrets
ls -alt ${WORKING_DIRECTORY}

cat > ${WORKING_DIRECTORY}/secrets/rsyncd.secrets <<EOF
# Username must match RSYNCD_AUTH_USER in .env
synobackup:replace-with-a-long-random-password
EOF

In [ ]:
# Run
docker compose --progress plain --file ../docker-compose.yaml --env-file ../docker-compose.example.env up --detach --build

In [87]:
# Run
WORKING_DIRECTORY=/nas-01/volume1/docker/rsync-server
cd ${WORKING_DIRECTORY}
docker compose --progress plain --file docker-compose.yaml --env-file docker-compose.nas-03_unraid.env up --detach --build

 Image rsyncd-server-rsyncd Building 
#1 [internal] load local bake definitions
#1 reading from stdin 534B done
#1 DONE 0.0s

#2 [internal] load build definition from dockerfile
#2 transferring dockerfile:
#2 transferring dockerfile: 246B done
#2 DONE 0.3s

#3 [internal] load metadata for docker.io/library/alpine:3.22
#3 DONE 0.5s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [1/4] FROM docker.io/library/alpine:3.22@sha256:55ae5d250caebc548793f321534bc6a8ef1d116f334f18f4ada1b2daad3251b2
#5 DONE 0.0s

#6 [internal] load build context
#6 transferring context: 66B done
#6 DONE 0.0s

#7 [3/4] COPY build/entrypoint.sh /usr/local/bin/entrypoint.sh
#7 CACHED

#8 [2/4] RUN apk add --no-cache rsync
#8 CACHED

#9 [4/4] RUN chmod 0755 /usr/local/bin/entrypoint.sh
#9 CACHED

#10 exporting to image
#10 exporting layers done
#10 writing image sha256:4814947b4cf17e8dfc028ee74881191fd886cf7990da5f28c99d8ae0b6907d2a done
#10 naming to docker.io/library/rsyncd-serve

In [90]:
# List Rsync Modules
rsync rsync://nas-03.internal

rsync: [Receiver] failed to connect to nas-03.internal (192.168.10.53): Connection refused (111)
rsync error: error in socket IO (code 10) at clientserver.c(139) [Receiver=3.2.7]


: 10

In [ ]:
# Generate User Passwords
printf 'username:%s\n' "$(openssl rand -base64 64 | head -c 32 | tr -d '\n' | tr '+/' '-_')"
openssl rand -base64 64 | tr -dc 'A-Za-z0-9' | head -c 64

In [ ]:
COUNT=12
LENGTH=64
FILTER='A-Za-z0-9' # alnum:'A-Za-z0-9'  # low:'a-z0-9'  # hex:'A-Fa-f0-9'

printf 'Password Generator | COUNT=%s | LENGTH=%s | FILTER=%s\n' \
  "$COUNT" "$LENGTH" "$FILTER"

for ((i=1; i<=COUNT; i++)); do
  openssl rand -base64 "$((LENGTH + 12))" | tr -dc 'A-Za-z0-9' | head -c "$LENGTH"
  echo
done

In [ ]:
# Create the Minecraft Bedrock Server & Backup Service
CONTEXT=nas-03
# docker --context lxc-docker-01 compose --profile "*" --env-file .env.lxc-docker  up --build --detach --remove-orphans

# Build the Base Image for Minecraft Bedrock Server Backup (Bedrockifier)
docker --context "${CONTEXT}" build \
  --progress plain \
  --tag local/bedrockifier-base:latest \
  --file ../repositories/minecraft-bedrock-backup/Docker/Dockerfile \
  ../repositories/minecraft-bedrock-backup

# Build + Run the Minecraft Bedrock Server & Backup Service
docker --context "${CONTEXT}" compose \
  --ansi never \
  --env-file ../.env.lxc-docker \
  up --build --detach --remove-orphans